### Try to find URLs for zotero entries that are missing them

Perplexity dialog markdown has references that are described only by URL so I have to use urls as a key to find the matching
entry in the zotero DB.  But about 150 of 1700 entries are missing URLs (jan 2025), and many are the 
new ones I want for refwrangle.

I tried getting the URLs by having perplexity read a csv file of items w/ no URL, and this was hopeless.
It (gpt01) could do the job, but kept trying to quit early or to shrug it off.  Super annoying.

So I tried to find titles for the perplexity output URLs and then match by title.  This was
far more successful, but I had a lot of problem with websites not responding to my queries.
Maybe they have screen scraping prevention of some kind...

Then I realized that the output of the **Save my ChatGPT extension** actually has (partial) titles 
in its Sources section.  I think I'll just use that extension 
**instead of messing with raw perplexity output**

In [12]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import sys
from icecream import ic
import requests
from bs4 import BeautifulSoup
from requests.exceptions import Timeout, RequestException
import time

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
outfile = rfw.refwrangle_test_dir / 'tmp' / 'entries_no_url.csv' # goes to perplexity gpto1

### Find entries with missing URLs

In [3]:

zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)
parentItems = zot.everything(zot.top())
def make_entry_row(parent):
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)
    cdict = {column:pdat.get(column) for column in ['date', 'itemType']}
    
    return cdict | dict(citekey=citekeyThis, zotkey=parent['key'], 
                     author = rfw.get_creator(parent), title=rfw.get_title(parent))

no_url_entries, url_entries = [], []
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    if (itemType := pdat['itemType']) == 'note':
        continue

    item_row = make_entry_row(parent)

    if url := pdat.get('url'): 
        url_entries.append(item_row | {'url':url})
    else:
        no_url_entries.append(item_row)

no_url_entries = pd.DataFrame(no_url_entries)
print(f"\n{len(no_url_entries)=} of {len(parentItems)} entries are missing URls\n")
url_entries = pd.DataFrame(url_entries)
ic(outfile)
no_url_entries.to_csv(outfile)
display(no_url_entries.head(), url_entries.head())


len(no_url_entries)=150 of 1685 entries are missing URls



ic| outfile: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/entries_no_url.csv')


,date,itemType,citekey,zotkey,author,title
0,March 23 2023,report,MMSDataModelSummary_v5.2,8XJHRYMU,Unknown Author,MMS Data Model Package Summary v5.2
1,1999,document,Seals99irradFrcstDiag,WYP9J7EU,"Seals, R",The heart of suny irradiance forecasting
2,2013,journalArticle,LaPaglia13TestIncrsSuggestibility,LDTF7M3L,"LaPaglia, Jessica A.",Testing increases suggestibility for narrative...
3,2020,journalArticle,Gaur20attribModellingRvw,HLHKVCLX,"Gaur, Jitendra",Attribution modelling in marketing: Literature...
4,2019,journalArticle,Wang19predOptcoolLdFrcst,R7TJLE7Y,"Wang, Lan",Cooling load forecasting-based predictive opti...


,date,itemType,citekey,zotkey,author,title,url
0,2009,journalArticle,Gerber09normsMotiveVote,6V73VWME,"Gerber, Alan S",Descriptive social norms and motivation to vot...,https://www.journals.uchicago.edu/doi/abs/10.1...
1,2013-08,thesis,Poston13politAdsVizAuralMeaning,6VPD3STC,"Poston, James Lake",Political advertising in the 2012 presidential...,https://scholarworks.boisestate.edu/td/602/
2,2021-11,thesis,Deaton21altruisByCountry,YIN5FL7M,"Deaton, Eddie William",An exploration of global altruistic variations...,https://rave.ohiolink.edu/etdc/view?acc_num=xu...
3,2017,conferencePaper,Calafiore17newsTopicSparseLrnPresid,KBL5IGVD,"Calafiore, Giuseppe C.",Topic analysis in news via sparse learning: a ...,https://www.sciencedirect.com/science/article/...
4,2023,book,Jarvis23gutenbergInternet,ETBJL3D9,"Jarvis, Jeff",The Gutenberg Parenthesis: The age of print an...,https://www.bloomsbury.com/us/gutenberg-parent...


### Try to get article titles from URLs
This isn't with AI, but doing my own web page requests, followed by old school string-matching.  

I'm testing this on zotero items that actually have titles, so I can see how well this could work.

In [4]:
def get_webpage_title(url, timeout=5, max_attempts=5, verbose=False):
    """Get the title of the page with a given URL"""
    for attempt in range(max_attempts):
        try:
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()  # Raises an HTTPError for bad responses
            soup = BeautifulSoup(response.content, 'html.parser')
            return soup.title.string.strip() if soup.title else None
        except Timeout:
            if verbose:
                print(f"Request timed out in {timeout=} seconds for URL: {url} (Attempt {attempt + 1}/{max_attempts})")
        except RequestException as e:
            if verbose:
                print(f"An error occurred while fetching URL {url}: {e} (Attempt {attempt + 1}/{max_attempts})")
        
        if attempt < max_attempts - 1:
            time.sleep(2 ** attempt)  # Exponential backoff
    if verbose:
        print(f"Failed to fetch title for URL: {url} after {max_attempts} attempts")
    
    return None

### Test web/zotero title matching
On zotero entries where I already have the URL and title, so I know what the right answer is.

**The title string matching algorithm is tested in match_titles_test.ipynb**

*On the 1st and only test, I'm shooting 5 out of 10 successes*

In [22]:
best_score_threshold = 70
max_tests=10
results = []
for _, search_item in url_entries.copy().iterrows(): # test items supplying the URLs you're looking for 
    if (ntests := len(results)) >= max_tests:
        print(f'Done testing {max_tests=}')
        break
    ic(ntests, search_item.zotkey, search_item.title)
    found_web_title = get_webpage_title(search_item.url)
    search_item['found_web_title'] = found_web_title
    search_item['is_match'] = False
    if found_web_title:
        ic(found_web_title)
        match_item, score = rfw.best_zotero_title_match(found_web_title, zot)
        search_item['key_best_zot'] = match_item['key']
        search_item['match_score'] = score
        search_item['is_match'] = score > best_score_threshold
        search_item['best_zot_title']=match_item['data'].get('title')
        if score > best_score_threshold:
            ic(search_item['best_zot_title'])
    results.append(search_item)

results = pd.DataFrame(results)

ic| ntests: 0
    search_item.zotkey: '6V73VWME'
    search_item.title: ("Descriptive social norms and motivation to vote: Everybody's voting and so "
                        'should you')
ic| ntests: 1
    search_item.zotkey: '6VPD3STC'
    search_item.title: ('Political advertising in the 2012 presidential election: How visual and '
                        'aural techniques are used to convey meaning')
ic| found_web_title: ('"Political Advertising in the 2012 Presidential Election: How Visual an" by '
                      'James Lake Poston')
ic| search_item['best_zot_title']: ('Political advertising in the 2012 presidential election: How visual and '
                                    'aural techniques are used to convey meaning')
ic| ntests: 2
    search_item.zotkey: 'YIN5FL7M'
    search_item.title: 'An exploration of global altruistic variations by country'
ic| found_web_title: 'OhioLINK ETD: Deaton, Eddie W.'
ic| ntests: 3
    search_item.zotkey: 'KBL5IGVD'
    search_item.tit

Done testing max_tests=10


In [24]:
results = pd.DataFrame(results)
results[['title', 'best_zot_title', 'is_match', 'match_score', 'found_web_title', 'citekey', 'key_best_zot']]

,title,best_zot_title,is_match,match_score,found_web_title,citekey,key_best_zot
0,Descriptive social norms and motivation to vot...,NaN,False,NaN,None,Gerber09normsMotiveVote,NaN
1,Political advertising in the 2012 presidential...,Political advertising in the 2012 presidential...,True,100.0,"""Political Advertising in the 2012 Presidentia...",Poston13politAdsVizAuralMeaning,6VPD3STC
2,An exploration of global altruistic variations...,Mutual Information between Discrete Variables ...,False,58.0,"OhioLINK ETD: Deaton, Eddie W.",Deaton21altruisByCountry,HGEZHK58
3,Topic analysis in news via sparse learning: a ...,NaN,False,NaN,None,Calafiore17newsTopicSparseLrnPresid,NaN
4,The Gutenberg Parenthesis: The age of print an...,NaN,False,NaN,None,Jarvis23gutenbergInternet,NaN
5,"Wrong: How media, politics, and identity drive...","Wrong: How media, politics, and identity drive...",True,100.0,Wrong | Hopkins Press,Young23wrongAppetiteMisinfo,EVVT7PL7
6,High conflict: Why we get trapped and how we g...,High conflict: Why we get trapped and how we g...,True,100.0,High Conflict | Book by Amanda Ripley | Offici...,Ripley22conflictTrapOut,UTU4PRME
7,Cheap speech: How disinformation poisons our p...,Cheap speech: How disinformation poisons our p...,True,100.0,Cheap Speech,Hasen22cheapSpchDisinfoCure,QMA5DB7E
8,Taking Action on Attention I,NaN,False,NaN,None,IAS23takingActionAttentionVI,NaN
9,Taking Action on Attention II,NaN,False,NaN,None,IAS24takingActionAttentionVII,NaN
